# Pooled 3-Class Benchmark (train / val / test folds)

Benchmarks every feature family on the **pooled 1,966-sequence set** (Waymo train + val, 3-class)
using the **frozen fold assignment** in `data/cnn_2d_seq_fold.csv` — the same split Matt's CNN uses.

**Protocol:** fit on the **train** fold (1,376), select the winning feature-set/model on the
**val** fold (295), report final numbers on the **test** fold (295). Scalers are fit on train only.
Every family is joined by **seq_id**, so nothing depends on row position.

**Families** (each pooled from its training array + the `*_val` array produced by
`scripts/extract_val_features.py`): HOG, HSV, YOLO, Road_v1, Road_v2, VehicleOcc, Trend+Flow.
If `CNN_2D.npy` (Matt's penultimate) and/or `av_embedding.npy` (AV-domain) are present, they're
added automatically and included in the "with vs without" comparison.

> **Note on the CNN feature:** the CNN was trained on the train fold, so its features on the
> *train* rows are in-sample for it — judge any improvement on **val/test**, where they're honest.
> All prior headline numbers (HOG 0.601 / Trend+Flow 0.648) were on the *old* 75/25 split of 1,604,
> so they are recomputed here on the new split and are not directly comparable to those.

## 0. Setup

In [ ]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

current_dir = Path.cwd()
PROJECT_ROOT = current_dir.parent if current_dir.name == 'notebooks' else current_dir
FD   = PROJECT_ROOT / 'data' / 'processed' / 'waymo_e2e' / 'features'
FOLD_CSV = PROJECT_ROOT / 'data' / 'cnn_2d_seq_fold.csv'

CLASSES = ['straight', 'right-turn', 'left-turn']
RS = 42

FOLD  = pd.read_csv(FOLD_CSV).sort_values('cnn_row').reset_index(drop=True)
ORDER = FOLD['seq_id'].astype(str).tolist()        # canonical pooled order (== CNN_2D.npy row order)
y     = FOLD['class_label'].to_numpy()
foldv = FOLD['fold'].to_numpy()
tr, va, te = foldv == 'train', foldv == 'val', foldv == 'test'
print(f'pool={len(ORDER)}  train={tr.sum()}  val={va.sum()}  test={te.sum()}')
print('class balance (pool):', dict(pd.Series(y).value_counts().reindex(CLASSES)))

## 1. Assemble every feature family by seq_id

Each family is built from its **training array** (keyed by its own seq_ids) plus the matching
`*_val` array (keyed by `seq_ids_val`), then laid out in the canonical pooled order. A family is
skipped with a warning if any file is missing.

In [ ]:
def _load(name):
    return np.load(FD / name, allow_pickle=True)

def two_source(tr_arr, tr_sid, va_arr, va_sid):
    A = _load(tr_arr); ts = _load(tr_sid).astype(str)
    B = _load(va_arr); vs = _load(va_sid).astype(str)
    d = {s: A[i] for i, s in enumerate(ts)}
    for i, s in enumerate(vs): d[s] = B[i]          # val rows
    missing = [s for s in ORDER if s not in d]
    if missing:
        raise KeyError(f'{len(missing)} pool seqs have no feature row (e.g. {missing[:2]})')
    return np.stack([np.asarray(d[s], dtype=np.float32) for s in ORDER])

# family_name -> (train_arr, train_seqids, val_arr, val_seqids)
SOURCES = {
    'HOG':        ('hog.npy',                    'seq_ids.npy',                    'hog_val.npy',                'seq_ids_val.npy'),
    'HSV':        ('hsv.npy',                    'seq_ids.npy',                    'hsv_val.npy',                'seq_ids_val.npy'),
    'YOLO':       ('yolo.npy',                   'seq_ids.npy',                    'yolo_val.npy',               'seq_ids_val.npy'),
    'Road_v1':    ('road.npy',                   'seq_ids.npy',                    'road_val.npy',               'seq_ids_val.npy'),
    'Road_v2':    ('road_v2_3class.npy',         'seq_ids_3class.npy',             'road_v2_val.npy',            'seq_ids_val.npy'),
    'VehicleOcc': ('vehicle_occupancy_3class.npy','seq_ids_3class.npy',            'vehicle_occupancy_val.npy',  'seq_ids_val.npy'),
    'Trend+Flow': ('framediff_v2_3class.npy',    'framediff_v2_3class_seq_ids.npy','framediff_v2_val.npy',       'seq_ids_val.npy'),
}

FAM = {}
for name, (ta, ts, vaa, vs) in SOURCES.items():
    try:
        FAM[name] = two_source(ta, ts, vaa, vs)
        print(f'  {name:12s} {FAM[name].shape}')
    except FileNotFoundError as e:
        print(f'  {name:12s} SKIPPED (missing file: {Path(str(e)).name})')

# ---- optional learned-feature families (single array in CNN-row order) ----
def single_source_cnnorder(arr_name):
    A = _load(arr_name)
    if len(A) != len(ORDER):
        raise ValueError(f'{arr_name}: {len(A)} rows != pool {len(ORDER)}')
    return np.asarray(A, dtype=np.float32)          # already in cnn_row order == ORDER

if (FD / 'CNN_2D.npy').exists():
    FAM['CNN'] = single_source_cnnorder('CNN_2D.npy')
    print(f'  {"CNN":12s} {FAM["CNN"].shape}  (Matt penultimate)')
    if (FD / 'labels_CNN_2D.npy').exists():
        lc = _load('labels_CNN_2D.npy').astype(str)
        # labels_CNN_2D is integer-encoded in some exports; only cross-check if it is string labels
        if lc.dtype.kind in 'US' and set(lc[:5]).issubset(set(CLASSES)):
            print('   labels_CNN_2D matches fold labels:', bool((lc == y).all()))
else:
    print('  CNN          not present yet — run scripts/gen_cnn2d_split_csvs.py then CNN_2D.ipynb')

if (FD / 'av_embedding.npy').exists():
    # av_embedding is keyed by its own seq_ids -> join by seq_id
    A = _load('av_embedding.npy'); s = _load('av_embedding_seq_ids.npy').astype(str)
    d = {sid: A[i] for i, sid in enumerate(s)}
    if all(sid in d for sid in ORDER):
        FAM['AV'] = np.stack([np.asarray(d[sid], np.float32) for sid in ORDER])
        print(f'  {"AV":12s} {FAM["AV"].shape}  (AV-domain embedding)')

print('\nfamilies available:', list(FAM))

## 2. Benchmark — fit on train, select on val, report on test

In [ ]:
def bench(combo):
    X  = np.concatenate([FAM[f] for f in combo], axis=1)
    sc = StandardScaler().fit(X[tr])
    Xtr, Xva, Xte = sc.transform(X[tr]), sc.transform(X[va]), sc.transform(X[te])
    row = {'features': ' + '.join(combo), 'dims': X.shape[1], '_combo': combo}
    for mname in ('SVM', 'RF'):
        if mname == 'RF':   # trees are scale-invariant -> raw, mirrors the v3 benchmark
            m = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RS, n_jobs=-1)
            m.fit(X[tr], y[tr]); pv, pt = m.predict(X[va]), m.predict(X[te])
        else:
            m = SVC(kernel='rbf', class_weight='balanced', random_state=RS)
            m.fit(Xtr, y[tr]); pv, pt = m.predict(Xva), m.predict(Xte)
        row[f'{mname}_val_f1']  = f1_score(y[va], pv, labels=CLASSES, average='macro')
        row[f'{mname}_test_f1'] = f1_score(y[te], pt, labels=CLASSES, average='macro')
        row[f'{mname}_test_acc']= accuracy_score(y[te], pt)
    return row

has = lambda *fs: all(f in FAM for f in fs)
combos = [[f] for f in FAM]                                  # every family alone
combos += [c for c in [
    ['HOG','Trend+Flow'],
    ['HOG','HSV','YOLO','Road_v2','VehicleOcc','Trend+Flow'],           # all classical
] if has(*c)]
if 'CNN' in FAM:   # ['CNN'] alone is already in the per-family loop above — don't re-add it
    combos += [c for c in [
        ['Trend+Flow','CNN'],
        ['HOG','Trend+Flow','CNN'],
        ['HOG','HSV','YOLO','Road_v2','VehicleOcc','Trend+Flow','CNN'],  # all classical + CNN
    ] if has(*c)]

rows = []
for i, c in enumerate(combos):
    print(f'  [{i+1}/{len(combos)}] {" + ".join(c)} ...', flush=True)   # progress (HOG combos are slow)
    rows.append(bench(c))
results = pd.DataFrame(rows)

maj = pd.Series(y[tr]).value_counts(normalize=True).max()
print(f'\nmajority-class baseline (train prior): {maj:.3f}\n')
show = results.drop(columns=['_combo']).sort_values('RF_test_f1', ascending=False).reset_index(drop=True)
pd.set_option('display.width', 200)
print(show.to_string(index=False,
      formatters={c: '{:.3f}'.format for c in show.columns if c.endswith(('_f1','_acc'))}))

## 3. Final model (best val among CNN-free) → held-out test, plus CNN ablation

In [ ]:
# The CNN used the val fold for early stopping/tuning, so val is not a fair basis for judging
# CNN combos. Select the FINAL model by val among CNN-FREE combos; treat CNN as an ablation.
def has_cnn(cmb): return 'CNN' in cmb
best = None  # (val_f1, combo_list, model)
for _, r in results.iterrows():
    if has_cnn(r['_combo']): continue
    for mname in ('SVM', 'RF'):
        cand = (r[f'{mname}_val_f1'], r['_combo'], mname)
        if best is None or cand[0] > best[0]: best = cand
val_f1, combo, mdl = best
feats = ' + '.join(combo)
print(f'Final model (best val among CNN-free): {feats} / {mdl}  (val f1={val_f1:.3f})')

def fit_predict_test(cmb, model):
    X = np.concatenate([FAM[f] for f in cmb], axis=1)
    if model == 'RF':
        m = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RS, n_jobs=-1).fit(X[tr], y[tr])
        return m.predict(X[te])
    sc = StandardScaler().fit(X[tr])
    m = SVC(kernel='rbf', class_weight='balanced', random_state=RS).fit(sc.transform(X[tr]), y[tr])
    return m.predict(sc.transform(X[te]))

pt = fit_predict_test(combo, mdl)
test_f1 = f1_score(y[te], pt, labels=CLASSES, average='macro')
print(f'\nTEST accuracy={accuracy_score(y[te],pt):.3f}  macro-F1={test_f1:.3f}  (majority baseline {maj:.3f})\n')
print(classification_report(y[te], pt, labels=CLASSES, zero_division=0))

# --- CNN ablation on the SAME base, judged on TEST (the only fold the CNN never used) ---
if 'CNN' in FAM:
    ptc = fit_predict_test(combo + ['CNN'], mdl)
    f1c = f1_score(y[te], ptc, labels=CLASSES, average='macro')
    print(f'CNN ablation ({mdl}, test macro-F1):  {feats} = {test_f1:.3f}   {feats} + CNN = {f1c:.3f}   '
          f'delta = {f1c - test_f1:+.3f}   (~0.05 = 1 SE on n={te.sum()})')

cm = confusion_matrix(y[te], pt, labels=CLASSES)
fig, ax = plt.subplots(figsize=(4.5,4))
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(3)); ax.set_xticklabels(CLASSES, rotation=30, ha='right')
ax.set_yticks(range(3)); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(f'Test — {feats} / {mdl}')
for i in range(3):
    for j in range(3):
        ax.text(j,i,cm[i,j],ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.tight_layout(); plt.show()

## 4. Read-out

- **Final model = best val macro-F1 among CNN-free combos**, reported on test (cell 3). The CNN
  is excluded from *selection* and shown only as an ablation — see why below.
- **The CNN used the val fold for early stopping and hyperparameter tuning**, so its penultimate
  features are optimistically biased on val, and val overstates the CNN's contribution. **Test is
  the only fold the CNN never saw** — so judge the CNN there, via the paired ablation (base vs
  base + CNN on the test column), not on val.
- With a 295-sequence test set, a ~0.05 macro-F1 gap is roughly **one standard error** — read the
  CNN ablation delta as "no clear test gain," not proof of no value. A fully fair test would use
  **out-of-fold** CNN features; the current in-sample train features actually bias *against* the
  CNN, so the honest claim is "did not improve held-out test in this stacking setup."
- Note: rbf-SVM on the raw 5,796-dim HOG is degenerate here (near-identical test F1 across all
  HOG combos) — trust the **RF** column for HOG rows, or PCA-reduce HOG before the SVM.
- If you run the AV-domain embedding notebook, `av_embedding.npy` appears as an `AV` family
  automatically; add it to `combos` to include it.

## 5. MoViNet stack for the class update — current best vs. Trend+Flow + MoViNet

Frozen **MoViNet-A0 (stream, Kinetics-600)** embeddings, cached by `CNN_3D_v2.ipynb`, stacked with
the Trend+Flow temporal features. These are the *frozen backbone* embeddings (leakage-free) — not the
trained head's `dense_hidden`. Files live in
`data/processed/waymo_e2e/CNN_3D_v2/embedding_cache/movinet_a0_stream_{train,val,test}_embeddings.npy`
(gitignored — pull from the team Drive, like the other feature arrays). This section is self-contained
and does not change the benchmark/selection above; it just renders the two confusion matrices we present.

In [ ]:
# --- frozen MoViNet-A0 embedding, laid out in the pooled ORDER (by seq_id) ---
MOVI_DIR = PROJECT_ROOT / 'data' / 'processed' / 'waymo_e2e' / 'CNN_3D_v2' / 'embedding_cache'

def load_movinet(order):
    d = {}
    for split in ('train', 'val', 'test'):
        arr  = _load_npy(MOVI_DIR / f'movinet_a0_stream_{split}_embeddings.npy')
        sids = FOLD[FOLD['fold'] == split].sort_values('cnn_row')['seq_id'].astype(str).tolist()
        assert len(sids) == len(arr), f'{split}: {len(sids)} fold seqs vs {len(arr)} embedding rows'
        d.update(zip(sids, arr))
    missing = [s for s in order if s not in d]
    assert not missing, f'{len(missing)} pool seqs have no MoViNet row (e.g. {missing[:2]})'
    return np.stack([np.asarray(d[s], np.float32) for s in order])

def _load_npy(p):
    return np.load(p, allow_pickle=True)

MOVI = load_movinet(ORDER)
print('MoViNet embedding:', MOVI.shape)

def _fit_predict_test(X, model):
    if model == 'RF':
        m = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                   random_state=RS, n_jobs=-1).fit(X[tr], y[tr])
        return m.predict(X[te])
    sc = StandardScaler().fit(X[tr])
    m = SVC(kernel='rbf', class_weight='balanced', random_state=RS).fit(sc.transform(X[tr]), y[tr])
    return m.predict(sc.transform(X[te]))

TF = FAM['Trend+Flow']
PANELS = [
    ('Trend+Flow / RF',            TF,                                  'RF'),
    ('Trend+Flow + MoViNet / SVM', np.concatenate([TF, MOVI], axis=1),  'SVM'),
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.7))
for ax, (title, X, mdl) in zip(axes, PANELS):
    p   = _fit_predict_test(X, mdl)
    f1  = f1_score(y[te], p, labels=CLASSES, average='macro')
    acc = accuracy_score(y[te], p)
    cm  = confusion_matrix(y[te], p, labels=CLASSES)
    cmn = cm / cm.sum(axis=1, keepdims=True)
    im  = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(3)); ax.set_xticklabels(CLASSES, rotation=30, ha='right')
    ax.set_yticks(range(3)); ax.set_yticklabels(CLASSES)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'{title}\nmacro-F1={f1:.3f}   acc={acc:.3f}', fontsize=10)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{cmn[i,j]*100:.0f}%\n({cm[i,j]})', ha='center', va='center',
                    color='white' if cmn[i,j] > 0.5 else '#222', fontsize=9)
    print(f'{title:30s} macro-F1={f1:.3f}  acc={acc:.3f}  '
          f'recall str/right/left = {cmn[0,0]:.2f}/{cmn[1,1]:.2f}/{cmn[2,2]:.2f}')

fig.suptitle('Held-out test set — current best vs. MoViNet stack', y=1.02, fontsize=11)
plt.tight_layout(); plt.show()